In [12]:
from src.utils.dataset_utils import count_dataset_stats
count_dataset_stats("dataset/fusion")

from src.modules.audio import audio_preprocess 
audio_preprocess.run()

from src.modules.audio import audio_train
audio_train.run()

KeyboardInterrupt: 

In [9]:
import os
import csv
import cv2
import numpy as np
from collections import defaultdict
from tqdm import tqdm
from ultralytics import YOLO

from src.modules.video.video_preprocess import video_preprocess
from src.modules.video import video_train
from src.config import HISTORY_LEN, SEQUENCE_LENGTH, STEP, CONSECUTIVE, FEATURE_DIM, JOINT_COUNT

ROOT_DATASET_DIR = "/Users/aleksandralitvak/Downloads/A-Dataset-for-Automatic-Violence-Detection-in-Videos-master 2/violence-detection-dataset"
YOLO_MODEL_PATH = "/Users/aleksandralitvak/Documents/codes/python/Cursova/assets/models/yolo11n-pose.pt"
AE_MODEL_PATH = "/Users/aleksandralitvak/Downloads/checkpoint_epoch_12.weights.h5"

VIOLENT_TAGS_CSV = os.path.join(ROOT_DATASET_DIR, "violent-action-classes.csv")
NONVIOLENT_TAGS_CSV = os.path.join(ROOT_DATASET_DIR, "nonviolent-action-classes.csv")
OUTPUT_CSV = "detailed_video_evaluation.csv"

VIDEO_EXTENSIONS = (".mp4", ".avi", ".mov", ".mkv")


def read_tags_csv(path):
    tags_by_file = {}

    with open(path, newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f, delimiter=";")
        reader.fieldnames = [name.strip() for name in reader.fieldnames]

        for row in reader:
            row = {k.strip(): v.strip() for k, v in row.items()}
            filename = row.get("FILE")
            tags_raw = row.get("ACTION CLASSES", "")

            if not filename:
                continue

            tags = [tag.strip() for tag in tags_raw.split(",") if tag.strip()]
            tags_by_file[filename] = tags

    return tags_by_file


def get_true_label(video_path):
    parts = [p.lower() for p in video_path.split(os.sep)]

    if "non-violent" in parts:
        return 0

    if "violent" in parts:
        return 1

    return None


def get_video_tags(video_path, violent_tags, nonviolent_tags):
    filename = os.path.basename(video_path)
    true_label = get_true_label(video_path)

    if true_label == 1:
        return violent_tags.get(filename, [])

    if true_label == 0:
        return nonviolent_tags.get(filename, [])

    return []


def calculate_iou(box1, box2, depth_thresh=0.5):
    y_bottom1 = box1[3]
    y_bottom2 = box2[3]
    h_max = max(box1[3] - box1[1], box2[3] - box2[1])

    if abs(y_bottom1 - y_bottom2) > h_max * depth_thresh:
        return 0.0

    x_left = max(box1[0], box2[0])
    y_top = max(box1[1], box2[1])
    x_right = min(box1[2], box2[2])
    y_bottom = min(box1[3], box2[3])

    if x_right < x_left or y_bottom < y_top:
        return 0.0

    intersection_area = (x_right - x_left) * (y_bottom - y_top)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])

    return intersection_area / float(box1_area + box2_area - intersection_area)


def test_single_video(video_path, yolo_model, ae_model, threshold):
    cap = cv2.VideoCapture(video_path)

    width_orig = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height_orig = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    if width_orig <= 0 or height_orig <= 0:
        cap.release()
        return False

    edge_margin_width = width_orig * 0.15
    edge_margin_height = height_orig * 0.15

    tracks_history = {}
    track_ages = {}
    alarm_triggered = False

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        results = yolo_model.track(
            frame,
            persist=True,
            verbose=False,
            stream=False,
            conf=0.5,
            iou=0.45,
            classes=[0],
            tracker="bytetrack.yaml",
            imgsz=640,
            half=False,
        )

        res = results[0]
        current_frame_ids = []

        if res.keypoints is None or len(res.keypoints.data) == 0 or res.boxes.id is None:
            continue

        boxes_all = res.boxes.xyxy.cpu().numpy()
        track_ids = res.boxes.id.int().cpu().tolist()
        kpts_all = res.keypoints.data.cpu().numpy()

        inputs_list = []
        targets_list = []
        inference_tracks = []

        for person_kpts, track_id, box in zip(kpts_all, track_ids, boxes_all):
            x1, y1, x2, y2 = box

            is_on_edge = (
                x1 < edge_margin_width or
                y1 < edge_margin_height or
                x2 > width_orig - edge_margin_width or
                y2 > height_orig - edge_margin_height
            )

            valid_points_count = 0
            missing_keypoints = False

            for pt in person_kpts:
                if pt[0] == 0.0 and pt[1] == 0.0:
                    missing_keypoints = True
                else:
                    valid_points_count += 1

            if (is_on_edge and missing_keypoints) or valid_points_count < 6:
                continue

            if track_id not in tracks_history:
                tracks_history[track_id] = []
                track_ages[track_id] = 0

            custom_pts = video_preprocess.get_yolo_mapping(person_kpts)
            tracks_history[track_id].append(custom_pts)
            track_ages[track_id] += 1
            current_frame_ids.append(track_id)

            if len(tracks_history[track_id]) > SEQUENCE_LENGTH:
                tracks_history[track_id].pop(0)

            if len(tracks_history[track_id]) == SEQUENCE_LENGTH:
                if (track_ages[track_id] - SEQUENCE_LENGTH) % STEP == 0:
                    seq = np.array(tracks_history[track_id])
                    norm_seq = video_preprocess.normalize_sequence(seq)
                    full_feat = video_preprocess.compute_kinematics(norm_seq)

                    inputs_list.append(full_feat[:HISTORY_LEN])
                    targets_list.append(full_feat[HISTORY_LEN:])
                    inference_tracks.append(track_id)

        if inputs_list:
            batch_x = np.array(inputs_list, dtype=np.float32)
            batch_y = np.array(targets_list, dtype=np.float32)

            preds = ae_model(batch_x, training=False)
            raw_loss_batch = video_train.get_loss_top3_joints(batch_y, preds).numpy()
            is_anomaly_batch = video_train.detect_sequence_anomaly(
                raw_loss_batch,
                threshold,
                consecutive=CONSECUTIVE,
            )

            for i, track_id in enumerate(inference_tracks):
                if is_anomaly_batch[i] != 1:
                    continue

                track_idx = track_ids.index(track_id)
                current_box = boxes_all[track_idx]
                max_iou = 0.0

                for j, other_box in enumerate(boxes_all):
                    if j == track_idx:
                        continue

                    iou = calculate_iou(current_box, other_box)
                    max_iou = max(max_iou, iou)

                if max_iou >= 0.01:
                    alarm_triggered = True
                    break

        tracks_history = {tid: h for tid, h in tracks_history.items() if tid in current_frame_ids}
        track_ages = {tid: a for tid, a in track_ages.items() if tid in current_frame_ids}

        if alarm_triggered:
            break

    cap.release()
    return alarm_triggered


def evaluate_dataset():
    violent_tags = read_tags_csv(VIOLENT_TAGS_CSV)
    nonviolent_tags = read_tags_csv(NONVIOLENT_TAGS_CSV)

    print("Завантаження моделей...")
    yolo_model = YOLO(YOLO_MODEL_PATH)

    A_norm = video_train.get_adjacency_matrix(JOINT_COUNT)
    ae_model = video_train.build_model(A_norm)
    ae_model.load_weights(AE_MODEL_PATH)

    threshold = video_train.load_threshold()
    print(f"Threshold: {threshold:.4f}")

    videos = []
    for root, _, files in os.walk(ROOT_DATASET_DIR):
        for filename in files:
            if filename.lower().endswith(VIDEO_EXTENSIONS):
                videos.append(os.path.join(root, filename))

    tp = fp = tn = fn = 0
    tag_stats = defaultdict(lambda: {"total": 0, "correct": 0, "tp": 0, "fp": 0, "tn": 0, "fn": 0})

    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["path", "filename", "true_label", "prediction", "correct", "tags"])

        for video_path in tqdm(videos, desc="Тестування відео"):
            true_label = get_true_label(video_path)

            if true_label is None:
                continue

            tags = get_video_tags(video_path, violent_tags, nonviolent_tags)
            prediction = 1 if test_single_video(video_path, yolo_model, ae_model, threshold) else 0
            correct = int(prediction == true_label)

            if true_label == 1 and prediction == 1:
                tp += 1
                result_key = "tp"
            elif true_label == 1 and prediction == 0:
                fn += 1
                result_key = "fn"
            elif true_label == 0 and prediction == 0:
                tn += 1
                result_key = "tn"
            else:
                fp += 1
                result_key = "fp"

            for tag in tags or ["NO_TAG"]:
                tag_stats[tag]["total"] += 1
                tag_stats[tag]["correct"] += correct
                tag_stats[tag][result_key] += 1

            writer.writerow([
                video_path,
                os.path.basename(video_path),
                true_label,
                prediction,
                correct,
                ",".join(tags),
            ])
            f.flush()

    total = tp + fp + tn + fn
    accuracy = (tp + tn) / total if total else 0
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    specificity = tn / (tn + fp) if (tn + fp) else 0

    print("\n--- ЗАГАЛЬНІ МЕТРИКИ ---")
    print(f"Загальна кількість: {total}")
    print(f"Загальна точність (Accuracy): {accuracy * 100:.2f}%")
    print(f"Точність для агресії (Precision): {precision * 100:.2f}%")
    print(f"Повнота для агресії (Recall): {recall * 100:.2f}%")
    print(f"Специфічність для норми (Specificity): {specificity * 100:.2f}%")

evaluate_dataset()

KeyboardInterrupt: 

In [ ]:
import csv
import os
from collections import defaultdict

INPUT_CSV = "/Users/aleksandralitvak/Documents/codes/python/Cursova/outputs/detailed_video_evaluation.csv"
TAG_TRANSLATION = {
    'hug': 'Обійми',
    'highfive': "Дати п'ять",
    'greet': 'Привітання',
    'handgestures': 'Активна жестикуляція',
    'jump': 'Стрибки',
    'handshake': 'Рукостискання',
    'walk': 'Ходьба',
    'slap': 'Удар по обличчю',
    'kick': 'Удар ногою',
    'club': 'Удар тупим предметом',
    'punch': 'Удар кулаком',
    'fight': 'Бійка',
    'choke': 'Удушення',
    'push': 'Поштовх',
    'stab': 'Удар ножем',
    'gunshot': 'Постріл'
}
FORBIDDEN_TAGS = {'club', 'stab', 'gunshot', 'friendly'}

def analyze_existing_csv():
    tp = fp = tn = fn = 0
    tag_stats = defaultdict(lambda: {"correct": 0, "total": 0})
    
    if not os.path.exists(INPUT_CSV):
        print(f"Файл {INPUT_CSV} не знайдено!")
        return

    with open(INPUT_CSV, mode='r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        
        for row in reader:
            tags = row['tags'].split(',')
            correct = int(row['correct'])
            
            for tag in tags:
                tag = tag.strip()
                if tag in FORBIDDEN_TAGS or tag == "NO_TAG":
                    continue
                
                tag_stats[tag]["total"] += 1
                tag_stats[tag]["correct"] += correct

                true_label = int(row['true_label'])
                prediction = int(row['prediction'])

                if true_label == 1 and prediction == 1:
                    tp += 1
                elif true_label == 1 and prediction == 0:
                    fn += 1
                elif true_label == 0 and prediction == 0:
                    tn += 1
                else:
                    fp += 1


    print("\n" + "="*70)
    print(f"{'Клас (Action Class)':<35} | {'Правильно':<10} | {'Всього':<8} | {'Точність'}")
    print("-" * 70)

    for tag, stats in tag_stats.items():
        display_name = TAG_TRANSLATION.get(tag, tag)
        total = stats["total"]
        if total > 0:
            accuracy = (stats["correct"] / total) * 100
            print(f"{display_name:<35} | {stats['correct']:<10} | {total:<8} | {accuracy:.2f}%")
    print("="*70)

    total = tp + fp + tn + fn
    accuracy = (tp + tn) / total if total else 0
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    specificity = tn / (tn + fp) if (tn + fp) else 0

    print("\n--- ЗАГАЛЬНІ МЕТРИКИ ---")
    print(f"Загальна кількість: {total}")
    print(f"Загальна точність (Accuracy): {accuracy * 100:.2f}%")
    print(f"Точність для агресії (Precision): {precision * 100:.2f}%")
    print(f"Повнота для агресії (Recall): {recall * 100:.2f}%")
    print(f"Специфічність для норми (Specificity): {specificity * 100:.2f}%")

if __name__ == "__main__":
    analyze_existing_csv()


Клас (Action Class)                 | Правильно  | Всього   | Точність
----------------------------------------------------------------------
Стрибки                             | 20         | 32       | 62.50%
Обійми                              | 29         | 68       | 42.65%
Привітання                          | 65         | 118      | 55.08%
Дати п'ять                          | 12         | 38       | 31.58%
Активна жестикуляція                | 34         | 64       | 53.12%
Поштовх                             | 19         | 22       | 86.36%
Удар по обличчю                     | 11         | 16       | 68.75%
Удар кулаком                        | 29         | 34       | 85.29%
Бійка                               | 40         | 40       | 100.00%
Удар ногою                          | 15         | 18       | 83.33%
Удушення                            | 14         | 14       | 100.00%
Рукостискання                       | 2          | 2        | 100.00%
Ходьба                    